[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_17_diffusion_sampling.ipynb)

# 🟡 Medium: Masked Diffusion Sampling Step

*Inference & Decoding*
Masked diffusion generates by starting from an all-`[MASK]` sequence and
repeatedly stepping backwards in time, revealing a few tokens each step.
Implement **one** such step: given $x_t$ and the model's logits, produce
$x_s$ for some $s < t$.

Because the forward kernel is absorbing and per-token, the exact posterior
factorises over positions and has no matrix in it:

$$q(x_s^i \mid x_t^i, x_0^i) =
\begin{cases}
\delta_{x_t^i} & x_t^i \neq \texttt{[MASK]}\\[4pt]
\dfrac{\alpha_s - \alpha_t}{1 - \alpha_t}\,\delta_{x_0^i}
\;+\; \dfrac{1 - \alpha_s}{1 - \alpha_t}\,\delta_{\texttt{[MASK]}}
& x_t^i = \texttt{[MASK]}
\end{cases}$$

Sampling replaces the unknown $x_0^i$ with a draw from the model. With the
linear schedule $\alpha_t = 1 - t$ the unmasking probability is just
$(t - s)/t$.

### Rules
- Signature: `denoise_step(key, logits, xt, t, s, *, mask_id)` -> `x_s`, shape of `xt`
- `logits` is `(B, L, V)`, `xt` is `(B, L)` integer ids, `t` and `s` are scalars
  with $0 \le s < t \le 1$
- Every position decides **independently** whether to unmask
- A position already holding a real token is **never** touched — not re-masked,
  not resampled
- A sampled token is never `[MASK]`
- Deterministic given `key`; use `jax.random.split` rather than passing one key
  to two draws
- `jax.nn.softmax` / `log_softmax` are allowed here; `optax` is not
- Must work under `jit`

### Carry-over unmasking, and why it is not optional
Once a token is revealed it stays. That is not a design choice bolted on for
convenience — it is what the absorbing posterior above *says*: the top branch is
a point mass on $x_t^i$. An implementation that resamples every position each
step is not sampling from this model at all; it will still produce fluent-looking
text, which is exactly what makes the bug survive review.

The practical consequence is that the number of masked tokens falls
monotonically, and at $s = 0$ the probability becomes $(t - 0)/t = 1$, so the
last step is guaranteed to leave nothing masked. You get termination for free
from the schedule rather than from a special case in the loop.

### The ratio, not the difference
$\alpha_s - \alpha_t$ is the *unconditional* mass that leaves the masked state
between the two times. But you are already conditioning on this position being
masked at $t$, an event of probability $1 - \alpha_t$, so the conditional rate
is the ratio $\frac{\alpha_s - \alpha_t}{1 - \alpha_t}$. Use the bare difference
and the sampler under-unmasks — badly at small $t$, where $1 - \alpha_t$ is
small — leaving `[MASK]` tokens in the output of a loop that thought it was done.

### One draw per position
`jax.random.bernoulli(key, p)` with no shape returns a **scalar**: every position
in the batch then makes the same decision, and the sampler reveals either
everything or nothing. Pass the shape explicitly. Same for the token draw —
`jax.random.categorical` reduces the last axis, so `(B, L, V)` logits give
`(B, L)` samples, one per position.

### What this leaves out
Real samplers add a *remasking* policy on top: rather than unmasking a random
subset, LLaDA-style decoding unmasks the positions the model is most confident
about, and some schedules deliberately re-mask low-confidence tokens later. That
last part steps outside the posterior above — it is where the discrete-flow-
matching view (which allows corrector steps) buys you something the plain
diffusion sampler cannot express. This problem is the exact posterior; the
policies are a layer above it.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def denoise_step(key, logits, xt, t, s, *, mask_id):
    """One reverse step of a masked diffusion LM: x_t -> x_s, with s < t.

    Args:
        key:     PRNG key
        logits:  (B, L, V) model scores for x0 given xt
        xt:      (B, L) current token ids, [MASK] where undecided
        t:       current time in (0, 1]
        s:       target time, 0 <= s < t
        mask_id: the [MASK] token id

    Returns:
        (B, L) token ids at time s.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp

V, MASK, L = 8, 7, 10

# A model that always wants token (position % 7), sharply.
logits = jax.nn.one_hot(jnp.arange(L) % 7, V)[None] * 30.0

# Start from all [MASK] and run the schedule down to 0.
x = jnp.full((1, L), MASK)
steps = 5
print("start:", x[0])
key = jax.random.key(0)
for i in range(steps):
    t, s = 1.0 - i / steps, 1.0 - (i + 1) / steps
    key, sub = jax.random.split(key)
    x = denoise_step(sub, logits, x, t, s, mask_id=MASK)
    print(f"t={t:.1f} -> s={s:.1f}:", x[0], f" masked: {int(jnp.sum(x == MASK))}")

print("\nno [MASK] left:", bool(jnp.all(x != MASK)))

# The unmasking rate is the RATIO, not the difference. Late in the schedule the
# two are wildly apart.
for t, s in [(1.0, 0.8), (0.4, 0.2), (0.1, 0.05)]:
    print(f"t={t}, s={s}:  ratio {(t - s) / t:.3f}   difference {t - s:.3f}")

# Carry-over: an already-decoded token survives any number of steps.
partial = jnp.array([[3, MASK, MASK, 3, MASK, 3, MASK, MASK, MASK, MASK]])
out = denoise_step(jax.random.key(1), logits, partial, 0.5, 0.25, mask_id=MASK)
print("\nbefore:", partial[0], "\nafter :", out[0])
print("decoded positions unchanged:",
      bool(jnp.all(out[partial != MASK] == partial[partial != MASK])))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("diffusion_sampling")

# hint("diffusion_sampling")      # stuck? nudge without the answer
# solution("diffusion_sampling")  # spoiler: the reference implementation
# status()                        # your dashboard across all problems